In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# Parsing: "1 0 2:45" means:
# freq 0 has amp=1
# freq 1 has amp=0
# freq 2 has amp=2, phase=45 degrees
#
# Optional explicit frequency:
# "2@5:30" means freq=5 amp=2 phase=30
# "1@-3" means freq=-3 amp=1 phase=0
# -------------------------
def parse_cycles(text: str):
    toks = [t for t in text.replace(",", " ").split() if t.strip()]
    cycles = []
    for i, tok in enumerate(toks):
        freq = i
        if "@" in tok:
            tok, f = tok.split("@", 1)
            freq = int(f)

        if ":" in tok:
            a, ph = tok.split(":", 1)
            amp = float(a)
            phase_deg = float(ph)
        else:
            amp = float(tok)
            phase_deg = 0.0

        if np.isnan(amp):
            continue
        cycles.append({"freq": freq, "amp": amp, "phase": np.deg2rad(phase_deg)})
    return cycles

def dft_from_time_samples(samples):
    x = np.array(samples, dtype=float)
    N = len(x)
    X = np.fft.fft(x) / N
    cycles = []
    for k in range(N):
        amp = np.abs(X[k])
        phase = np.angle(X[k])
        cycles.append({"freq": k, "amp": amp, "phase": phase})
    return cycles

def cycles_to_string(cycles, max_terms=16, amp_digits=2, phase_digits=0):
    out = []
    for c in cycles[:max_terms]:
        amp = round(c["amp"], amp_digits)
        ph = round(np.rad2deg(c["phase"]), phase_digits)
        if amp == 0:
            out.append("0")
        elif ph == 0:
            out.append(f"{amp}")
        else:
            out.append(f"{amp}:{ph}")
    return " ".join(out)

def phasor_sum(cycles, t):
    # total complex value at time t (radians)
    z = 0j
    parts = []
    for c in cycles:
        contrib = c["amp"] * np.exp(1j * (c["freq"] * t + c["phase"]))
        parts.append(contrib)
        z += contrib
    return z, parts

def basis_phasor(freq_k, t):
    # e^{-i k t} (FFT "spin backwards")
    return np.exp(-1j * freq_k * t)

# -------------------------
# Main renderer
# -------------------------
def render(
    input_mode,
    cycles_text,
    time_text,
    running,
    t_slider,
    show_total,
    show_parts,
    show_ticks,
    derive_mode,
    derive_k,
    N_ticks
):
    # Build cycles depending on mode
    if input_mode == "Cycles → Wave":
        cycles = parse_cycles(cycles_text)
        samples = None
    else:
        toks = [t for t in time_text.replace(",", " ").split() if t.strip()]
        samples = [float(t) for t in toks]
        if len(samples) < 2:
            cycles = [{"freq": 0, "amp": 0, "phase": 0}]
        else:
            cycles = dft_from_time_samples(samples)

    # Time setup
    steps = 180
    t = (t_slider / steps) * 2*np.pi

    # Discrete tick points
    N = int(N_ticks)
    ts = np.linspace(0, 2*np.pi, N, endpoint=False)

    # Continuous curve for wave
    fine = np.linspace(0, 2*np.pi, 720)
    y_total = np.array([phasor_sum(cycles, tt)[0].real for tt in fine])

    # Current phasors
    z, parts = phasor_sum(cycles, t)

    # ---- Plot ----
    plt.close("all")
    fig, (axC, axT) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios":[1, 2]})

    # --- Left: unit-circle style / phasors ---
    axC.set_aspect("equal", adjustable="box")
    axC.axhline(0, lw=1, alpha=0.35)
    axC.axvline(0, lw=1, alpha=0.35)

    # Choose a scale so it fits nicely
    max_amp = max([abs(c["amp"]) for c in cycles] + [1e-9])
    radius = max(1.0, min(3.0, max_amp))
    circle = plt.Circle((0,0), 1.0, fill=False, lw=1, alpha=0.35)
    axC.add_patch(circle)

    # Draw vectors tip-to-tail (epicycles)
    origin = 0+0j
    if show_parts:
        for contrib in parts:
            axC.plot([origin.real, (origin+contrib).real], [origin.imag, (origin+contrib).imag], lw=2, alpha=0.75)
            origin += contrib

    if show_total:
        axC.plot([0, z.real], [0, z.imag], lw=3, alpha=0.9)
        axC.scatter([z.real], [z.imag], s=45)

    axC.set_xlim(-radius, radius)
    axC.set_ylim(-radius, radius)
    axC.set_title("Rotating vectors (phasors)")
    axC.set_xticks([])
    axC.set_yticks([])

    # --- Right: time wave ---
    axT.axhline(0, lw=1, alpha=0.35)
    axT.plot(fine, y_total, lw=2, label="Total wave")

    # Show parts (each frequency's real contribution)
    if show_parts:
        for c in cycles[:12]:  # keep it readable
            part = np.array([(c["amp"] * np.exp(1j*(c["freq"]*tt + c["phase"]))).real for tt in fine])
            axT.plot(fine, part, lw=1, alpha=0.35)

    # Ticks (discrete samples)
    if show_ticks:
        ys = np.array([phasor_sum(cycles, tt)[0].real for tt in ts])
        axT.scatter(ts, ys, s=25, alpha=0.9)
        for tt in ts:
            axT.axvline(tt, lw=0.6, alpha=0.12)

    # Current time marker
    axT.axvline(t, lw=2, alpha=0.6)
    axT.scatter([t], [z.real], s=60)

    axT.set_xlim(0, 2*np.pi)
    axT.set_title("Wave over time (real part)")
    axT.set_xlabel("time (radians, 0 → 2π)")

    # --- Derive mode: show “spinning backwards” idea for one frequency ---
    if derive_mode:
        k = int(derive_k)
        # sample-based estimate (like DFT)
        ys = np.array([phasor_sum(cycles, tt)[0].real for tt in ts])
        basis = np.array([np.cos(k*tt) for tt in ts])  # beginner-friendly: cosine basis
        prod = ys * basis
        est = prod.mean()

        axT2 = axT.twinx()
        axT2.plot(ts, basis, marker="o", lw=1.2, alpha=0.5)
        axT2.plot(ts, prod, marker="o", lw=1.2, alpha=0.5)
        axT2.set_ylabel(f"Derive k={k}: basis & product (avg≈{est:.3f})")

    # Helpful caption
    if input_mode == "Time samples → Cycles (DFT)":
        axT.text(0.02, 0.95, "You entered time samples → we computed cycles (DFT).",
                 transform=axT.transAxes, va="top", alpha=0.75)

    plt.tight_layout()
    plt.show()

# -------------------------
# Widgets (UI)
# -------------------------
input_mode = widgets.ToggleButtons(
    options=["Cycles → Wave", "Time samples → Cycles (DFT)"],
    value="Cycles → Wave",
    description="Mode:"
)

cycles_text = widgets.Text(
    value="1 0 2:45",
    description="Cycles:",
    layout=widgets.Layout(width="420px")
)

time_text = widgets.Text(
    value="1 2 3 2 1 0 -1 0",
    description="Time:",
    layout=widgets.Layout(width="420px")
)

running = widgets.Checkbox(value=True, description="Running")
show_total = widgets.Checkbox(value=True, description="Total")
show_parts = widgets.Checkbox(value=True, description="Parts")
show_ticks = widgets.Checkbox(value=False, description="Ticks")
derive_mode = widgets.Checkbox(value=False, description="Derive")
derive_k = widgets.IntSlider(value=1, min=0, max=20, step=1, description="k")
N_ticks = widgets.IntSlider(value=16, min=4, max=64, step=1, description="Ticks N")

steps = 180
t_slider = widgets.IntSlider(value=0, min=0, max=steps, step=1, description="Time")

play = widgets.Play(interval=50, value=0, min=0, max=steps, step=1)
widgets.jslink((play, "value"), (t_slider, "value"))

out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        render(
            input_mode.value,
            cycles_text.value,
            time_text.value,
            running.value,
            t_slider.value,
            show_total.value,
            show_parts.value,
            show_ticks.value,
            derive_mode.value,
            derive_k.value,
            N_ticks.value
        )

# auto-advance if running
def tick(change):
    if running.value:
        play._playing = True
    else:
        play._playing = False

running.observe(tick, names="value")

for w in [input_mode, cycles_text, time_text, t_slider, show_total, show_parts, show_ticks, derive_mode, derive_k, N_ticks]:
    w.observe(refresh, names="value")

controls1 = widgets.HBox([input_mode])
controls2 = widgets.HBox([cycles_text, time_text])
controls3 = widgets.HBox([show_total, show_parts, show_ticks, derive_mode, derive_k, N_ticks])
controls4 = widgets.HBox([running, play, t_slider])

display(widgets.VBox([controls1, controls2, controls3, controls4, out]))
refresh()


: 